# Lab 04 - Linear Systems I

Solutions for all tasks from `L4_NC.pdf`, implemented in Python.

This notebook contains:
1. Back and forward substitution routines for triangular systems
2. Gaussian elimination with partial pivoting
3. Solvers based on LUP, Cholesky, and QR factorizations
4. The required applications and the optional structured-system application

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


def validate_square_matrix(A: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square matrix")

    return A.copy()


def validate_linear_system(A: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    A = validate_square_matrix(A)
    b = np.asarray(b, dtype=float).reshape(-1)

    if b.shape[0] != A.shape[0]:
        raise ValueError("b must have the same length as the dimension of A")

    return A, b.copy()


def back_substitution(U: np.ndarray, b: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    U, b = validate_linear_system(U, b)
    n = U.shape[0]
    x = np.zeros(n, dtype=float)

    for i in range(n - 1, -1, -1):
        if abs(U[i, i]) < tol:
            raise np.linalg.LinAlgError("Zero pivot encountered during back substitution")
        x[i] = (b[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]

    return x


def forward_substitution(L: np.ndarray, b: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    L, b = validate_linear_system(L, b)
    n = L.shape[0]
    x = np.zeros(n, dtype=float)

    for i in range(n):
        if abs(L[i, i]) < tol:
            raise np.linalg.LinAlgError("Zero pivot encountered during forward substitution")
        x[i] = (b[i] - L[i, :i] @ x[:i]) / L[i, i]

    return x


def gaussian_elimination_partial_pivot(
    A: np.ndarray,
    b: np.ndarray,
    tol: float = 1e-12,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    U, rhs = validate_linear_system(A, b)
    n = U.shape[0]

    for k in range(n - 1):
        pivot_row = k + np.argmax(np.abs(U[k:, k]))
        if abs(U[pivot_row, k]) < tol:
            raise np.linalg.LinAlgError("Matrix is singular to working precision")

        if pivot_row != k:
            U[[k, pivot_row], :] = U[[pivot_row, k], :]
            rhs[[k, pivot_row]] = rhs[[pivot_row, k]]

        for i in range(k + 1, n):
            multiplier = U[i, k] / U[k, k]
            U[i, k:] -= multiplier * U[k, k:]
            U[i, k] = 0.0
            rhs[i] -= multiplier * rhs[k]

    if abs(U[-1, -1]) < tol:
        raise np.linalg.LinAlgError("Matrix is singular to working precision")

    x = back_substitution(U, rhs, tol=tol)
    return x, U, rhs


def lup_factorization(A: np.ndarray, tol: float = 1e-12) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    U = validate_square_matrix(A)
    n = U.shape[0]
    L = np.eye(n, dtype=float)
    P = np.eye(n, dtype=float)

    for k in range(n - 1):
        pivot_row = k + np.argmax(np.abs(U[k:, k]))
        if abs(U[pivot_row, k]) < tol:
            raise np.linalg.LinAlgError("Matrix is singular to working precision")

        if pivot_row != k:
            U[[k, pivot_row], :] = U[[pivot_row, k], :]
            P[[k, pivot_row], :] = P[[pivot_row, k], :]
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]
            U[i, k] = 0.0

    if abs(U[-1, -1]) < tol:
        raise np.linalg.LinAlgError("Matrix is singular to working precision")

    return P, L, U


def solve_lup(A: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    A, b = validate_linear_system(A, b)
    P, L, U = lup_factorization(A)
    y = forward_substitution(L, P @ b)
    x = back_substitution(U, y)
    return x, P, L, U


def cholesky_upper(A: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    A = validate_square_matrix(A)

    if not np.allclose(A, A.T, atol=tol):
        raise ValueError("Cholesky factorization requires a symmetric matrix")

    n = A.shape[0]
    R = np.zeros_like(A)

    for k in range(n):
        diagonal_term = A[k, k] - np.dot(R[:k, k], R[:k, k])
        if diagonal_term <= tol:
            raise np.linalg.LinAlgError("Matrix is not positive definite")

        R[k, k] = np.sqrt(diagonal_term)

        for j in range(k + 1, n):
            numerator = A[k, j] - np.dot(R[:k, k], R[:k, j])
            R[k, j] = numerator / R[k, k]

    return R


def solve_cholesky(A: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    A, b = validate_linear_system(A, b)
    R = cholesky_upper(A)
    y = forward_substitution(R.T, b)
    x = back_substitution(R, y)
    return x, R


def qr_factorization(A: np.ndarray, tol: float = 1e-12) -> tuple[np.ndarray, np.ndarray]:
    A = validate_square_matrix(A)
    m, n = A.shape
    Q = np.zeros((m, n), dtype=float)
    R = np.zeros((n, n), dtype=float)
    V = A.copy()

    for k in range(n):
        R[k, k] = np.linalg.norm(V[:, k])
        if R[k, k] < tol:
            raise np.linalg.LinAlgError("Matrix is rank deficient")

        Q[:, k] = V[:, k] / R[k, k]

        for j in range(k + 1, n):
            R[k, j] = Q[:, k] @ V[:, j]
            V[:, j] -= R[k, j] * Q[:, k]

    return Q, R


def solve_qr(A: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    A, b = validate_linear_system(A, b)
    Q, R = qr_factorization(A)
    x = back_substitution(R, Q.T @ b)
    return x, Q, R


def residual_norm(A: np.ndarray, x: np.ndarray, b: np.ndarray) -> float:
    A, b = validate_linear_system(A, b)
    x = np.asarray(x, dtype=float).reshape(-1)
    return float(np.linalg.norm(A @ x - b))


def print_solution(method: str, A: np.ndarray, b: np.ndarray, x: np.ndarray) -> None:
    print(f"{method}:")
    print("x =", x)
    print(f"||Ax - b|| = {residual_norm(A, x, b):.3e}")
    print()


def solve_and_report(A: np.ndarray, b: np.ndarray) -> None:
    x_gauss, _, _ = gaussian_elimination_partial_pivot(A, b)
    print_solution("Gaussian elimination with partial pivoting", A, b, x_gauss)

    x_lup, P, L, U = solve_lup(A, b)
    print_solution("LUP factorization", A, b, x_lup)

    try:
        x_cholesky, R = solve_cholesky(A, b)
        print_solution("Cholesky factorization", A, b, x_cholesky)
    except (ValueError, np.linalg.LinAlgError) as exc:
        print("Cholesky factorization:")
        print(f"not applicable: {exc}")
        print()

    x_qr, Q, R = solve_qr(A, b)
    print_solution("QR factorization", A, b, x_qr)


def build_tridiagonal_system(n: int) -> tuple[np.ndarray, np.ndarray]:
    if n < 3:
        raise ValueError("n must be at least 3")

    A = 5.0 * np.eye(n)
    idx = np.arange(n - 1)
    A[idx, idx + 1] = -1.0
    A[idx + 1, idx] = -1.0

    b = np.full(n, 3.0)
    b[0] = 4.0
    b[-1] = 4.0
    return A, b


def build_banded_system(n: int) -> tuple[np.ndarray, np.ndarray]:
    if n < 7:
        raise ValueError("n must be at least 7")

    A = 5.0 * np.eye(n)
    for offset in (1, 3):
        idx = np.arange(n - offset)
        A[idx, idx + offset] = -1.0
        A[idx + offset, idx] = -1.0

    b = np.ones(n)
    b[0] = 3.0
    b[1] = 2.0
    b[2] = 2.0
    b[-3] = 2.0
    b[-2] = 2.0
    b[-1] = 3.0
    return A, b

## Exercise 1

Implement back substitution for upper triangular systems and forward substitution for lower triangular systems.

To verify the routines directly, the next cell solves one upper-triangular system and one lower-triangular system with known right-hand sides.

In [2]:
x_true = np.array([1.0, 2.0, -1.0])

U = np.array([
    [2.0, -1.0, 1.0],
    [0.0, 3.0, 2.0],
    [0.0, 0.0, 5.0],
])
b_upper = U @ x_true
x_upper = back_substitution(U, b_upper)

print("Upper triangular system:")
print("U =\n", U)
print("b =", b_upper)
print("Computed solution =", x_upper)
print(f"Residual norm = {residual_norm(U, x_upper, b_upper):.3e}")
print()

L = np.array([
    [2.0, 0.0, 0.0],
    [-1.0, 3.0, 0.0],
    [4.0, 2.0, 1.0],
])
b_lower = L @ x_true
x_lower = forward_substitution(L, b_lower)

print("Lower triangular system:")
print("L =\n", L)
print("b =", b_lower)
print("Computed solution =", x_lower)
print(f"Residual norm = {residual_norm(L, x_lower, b_lower):.3e}")

Upper triangular system:
U =
 [[ 2. -1.  1.]
 [ 0.  3.  2.]
 [ 0.  0.  5.]]
b = [-1.  4. -5.]
Computed solution = [ 1.  2. -1.]
Residual norm = 0.000e+00

Lower triangular system:
L =
 [[ 2.  0.  0.]
 [-1.  3.  0.]
 [ 4.  2.  1.]]
b = [2. 5. 7.]
Computed solution = [ 1.  2. -1.]
Residual norm = 0.000e+00


## Exercises 2 and 3

The helper routines above implement Gaussian elimination with partial pivoting and the requested factorization-based solvers. We now apply them to the systems from the assignment.

## Application 1

Solve the fixed `4 x 4` linear system from the statement using Gaussian elimination and all applicable factorizations.

Cholesky is only applicable to symmetric positive definite matrices, so the notebook checks that condition automatically.

In [3]:
A1 = np.array([
    [2.0, 1.0, -1.0, -2.0],
    [4.0, 4.0, 1.0, 3.0],
    [-6.0, -1.0, 10.0, 10.0],
    [-2.0, 1.0, 8.0, 4.0],
])
b1 = np.array([2.0, 4.0, -5.0, 1.0])

print("A =\n", A1)
print("b =", b1)
print()

solve_and_report(A1, b1)

A =
 [[ 2.  1. -1. -2.]
 [ 4.  4.  1.  3.]
 [-6. -1. 10. 10.]
 [-2.  1.  8.  4.]]
b = [ 2.  4. -5.  1.]

Gaussian elimination with partial pivoting:
x = [-2.375  4.25  -0.5   -1.   ]
||Ax - b|| = 2.512e-15

LUP factorization:
x = [-2.375  4.25  -0.5   -1.   ]
||Ax - b|| = 1.256e-15

Cholesky factorization:
not applicable: Cholesky factorization requires a symmetric matrix

QR factorization:
x = [-2.375  4.25  -0.5   -1.   ]
||Ax - b|| = 2.045e-14



## Application 2

Generate the tridiagonal system of arbitrary order `n >= 3`,

$$
5x_1 - x_2 = 4, \qquad -x_{j-1} + 5x_j - x_{j+1} = 3 \; (j = 2, \ldots, n-1), \qquad -x_{n-1} + 5x_n = 4,
$$

and solve it with the implemented methods. The generator below works for any valid `n`; here we illustrate it with `n = 8`.

In [4]:
n2 = 8
A2, b2 = build_tridiagonal_system(n2)

print(f"n = {n2}")
print("A =\n", A2)
print("b =", b2)
print()

solve_and_report(A2, b2)

n = 8
A =
 [[ 5. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  5. -1.  0.  0.  0.  0.  0.]
 [ 0. -1.  5. -1.  0.  0.  0.  0.]
 [ 0.  0. -1.  5. -1.  0.  0.  0.]
 [ 0.  0.  0. -1.  5. -1.  0.  0.]
 [ 0.  0.  0.  0. -1.  5. -1.  0.]
 [ 0.  0.  0.  0.  0. -1.  5. -1.]
 [ 0.  0.  0.  0.  0.  0. -1.  5.]]
b = [4. 3. 3. 3. 3. 3. 3. 4.]

Gaussian elimination with partial pivoting:
x = [1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.538e-15

LUP factorization:
x = [1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.538e-15

Cholesky factorization:
x = [1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.332e-15

QR factorization:
x = [1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.601e-15



## Optional Application 3

Generate the structured banded system of arbitrary order `n >= 7` with main diagonal equal to `5`, with `-1` on the first off-diagonals and also on the diagonals at distance `3`, and with right-hand side

$$
[3, 2, 2, 1, \ldots, 1, 2, 2, 3]^T.
$$

The generator below works for any valid `n`; here we illustrate it with `n = 9`.

In [5]:
n3 = 9
A3, b3 = build_banded_system(n3)

print(f"n = {n3}")
print("A =\n", A3)
print("b =", b3)
print()

solve_and_report(A3, b3)

n = 9
A =
 [[ 5. -1.  0. -1.  0.  0.  0.  0.  0.]
 [-1.  5. -1.  0. -1.  0.  0.  0.  0.]
 [ 0. -1.  5. -1.  0. -1.  0.  0.  0.]
 [-1.  0. -1.  5. -1.  0. -1.  0.  0.]
 [ 0. -1.  0. -1.  5. -1.  0. -1.  0.]
 [ 0.  0. -1.  0. -1.  5. -1.  0. -1.]
 [ 0.  0.  0. -1.  0. -1.  5. -1.  0.]
 [ 0.  0.  0.  0. -1.  0. -1.  5. -1.]
 [ 0.  0.  0.  0.  0. -1.  0. -1.  5.]]
b = [3. 2. 2. 1. 1. 1. 2. 2. 3.]

Gaussian elimination with partial pivoting:
x = [1. 1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.404e-15

LUP factorization:
x = [1. 1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.404e-15

Cholesky factorization:
x = [1. 1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 8.882e-16

QR factorization:
x = [1. 1. 1. 1. 1. 1. 1. 1. 1.]
||Ax - b|| = 1.601e-15

